# verify09: 機能2「~90-100長ベクトル → トリアージ」検証（vector_triage.py）

全質問ノードのBERT予測(code)を固定順ベクトルにまとめ、決定的にトリアージへ変換する機能の検証。
ローカル実行（BERTモデル不要・pure Python）。トリアージ変換の実体は protocol.yaml のルール探索。

前提: `vector_triage.py` / `triage_pipeline.py` / `age_logic.py` / `transition_diagram/protocol.yaml` が同じリポジトリにあること。

In [1]:
import importlib
import age_logic, triage_pipeline as tp, vector_triage as vt
for m in (age_logic, tp, vt): importlib.reload(m)
print('VECTOR_LEN =', vt.VECTOR_LEN, '（ベクトルの次元数＝トリアージ判定に使うBERTノード数）')

VECTOR_LEN = 105 （ベクトルの次元数＝トリアージ判定に使うBERTノード数）


## 1. ベクトルのスキーマ（各次元＝どのノードか）

In [2]:
import pandas as pd
desc = pd.DataFrame(vt.describe_vector())
print('先頭12次元:')
display(desc.head(12))
display(desc.tail(5))

先頭12次元:


,index,bert_node,yaml_nodes
0,0,00_common_breathing,[common_breathing]
1,1,00_common_cold_sweat,"[common_cold_sweat, common_cold_sweat_age_subq..."
2,2,00_common_conversation,[common_conversation]
3,3,00_common_face_color,[common_face_color]
4,4,00_observation,[observation]
5,5,00_overview,[overview]
6,6,03_chest_pain,[palpitation_chest_pain]
7,7,03_heart_history,[palpitation_heart_history]
8,8,03_icd_fired,[palpitation_icd_fired]
9,9,04_abdominal_pain,[syncope_abdominal_pain]


,index,bert_node,yaml_nodes
100,100,23_ped_head_loss_of_consciousness,[ped_head_loss_of_consciousness]
101,101,23_ped_head_neck_posture,[ped_head_neck_posture]
102,102,23_ped_head_nosebleed,[ped_head_nosebleed]
103,103,23_ped_head_visual_disturbance,[ped_head_visual_disturbance]
104,104,23_ped_head_vomiting_repeated,[ped_head_vomiting_repeated]


## 2. 変換の往復（build_vector ↔ vector_to_bertpred）

In [3]:
bp = {'14_numbness_stroke_symptoms':2, '14_numbness_stroke_history':3, '06_sudden_severe':1}
vec = vt.build_vector(bp)
back = vt.vector_to_bertpred(vec)
print('ベクトル長:', len(vec), ' 非ゼロ:', {k:v for k,v in back.items() if v})
assert len(vec)==vt.VECTOR_LEN
assert all(back[k]==v for k,v in bp.items())
print('build_vector <-> vector_to_bertpred: 往復OK')

ベクトル長: 105  非ゼロ: {'06_sudden_severe': 1, '14_numbness_stroke_history': 3, '14_numbness_stroke_symptoms': 2}
build_vector <-> vector_to_bertpred: 往復OK


## 3. ベクトル → トリアージ（フルの通報を再現）

共通バイタルを正常に流し、しびれ(脳卒中サイン→既往不明=R3)のベクトルを作って判定する。
導入の構造化入力（救急?等）は `base_answers` 既定を使用。年齢=70。

In [4]:
bp_full = {'00_observation':1,'00_common_breathing':1,'00_common_cold_sweat':2,
           '00_common_face_color':2,'00_common_conversation':1,
           '14_numbness_stroke_symptoms':2,'14_numbness_stroke_history':3}
vec = vt.build_vector(bp_full)
res = vt.vector_to_triage(vec, age=70)
print(res.report())
assert res.main == 'VE' and res.sub1 == 'R3'
print('OK: 共通完了 -> numbness完了(R3)')

■ メイン: Very Emergency  (main=VE)
■ サブ1(根拠コード): R3
■ なぜ: 共通完了 → numbness が完了(R3) をそのまま採用
■ 詳細: 共通完了かつ症状完了1つ(numbness:R3)
■ 共通フロー: 完了(症候別へ到達)  [reached:route_to_symptom_inquiry]
■ 完了した症候: numbness
OK: 共通完了 -> numbness完了(R3)


## 4. 別シナリオ: 共通完了だが全症候途切れ → VE（途切れ最長症候も出る）

In [5]:
bp2 = {'00_observation':1,'00_common_breathing':1,'00_common_cold_sweat':2,
       '00_common_face_color':2,'00_common_conversation':1,
       '14_numbness_stroke_symptoms':2}
res2 = vt.vector_to_triage(vt.build_vector(bp2), age=70)
print(res2.report())
assert res2.main == 'VE' and res2.furthest_broke_symptom == 'numbness'
print('OK: 共通完了 -> 全症候途切れ(VE) / 途切れ最長=numbness')

■ メイン: Very Emergency  (main=VE)
■ サブ1(根拠コード): 途切れVE(該当なし)
■ なぜ: 共通は完了したが全症候が途切れ → 判断根拠不足のため安全側で VE
■ 詳細: 共通完了＋症状全て途切れ
■ 共通フロー: 完了(症候別へ到達)  [reached:route_to_symptom_inquiry]
■ 途切れた中で最長遷移: numbness（2遷移）
■ 途切れ症候(遷移数降順): numbness(2)
OK: 共通完了 -> 全症候途切れ(VE) / 途切れ最長=numbness


## まとめ
- `VECTOR_LEN` ≒ 90-100（実測 105）。各次元は1つのBERT質問ノードの回答code。
- `build_vector`/`vector_to_bertpred` は可逆。
- `vector_to_triage(vector, age)` はベクトル＋年齢から、protocol.yaml 探索でメイン(VE/SE/LE)・サブ1(R/Y/G or 途切れVE)・なぜ・途切れ最長症候まで一括出力。
- 全 assert 通過なら機能2は検証OK。